# Classification d'images à l'aide d'algorithmes de Deep Learning

Projet n&#8239;$^\text{o}$ 6 du [cursus Machine Learning Engineer][2] d'OpenClassrooms

Auteur : [Kiril ISAKOV][1]

Mentor : Nicolas TISSERAND

Projet démarré le 23/03/2026

[1]: https://github.com/kirisakow/
[2]: https://openclassrooms.com/fr/paths/794-machine-learning-engineer

# Identifier la race d'un chien sur une photographie à l'aide d'un modèle *deep learning*

## Imports et constantes

In [1]:
DEFAULT_TRGT_IMG_SIZE = (224, 224)
DEFAULT_BATCH_SIZE = 32

import os
# Déclarer le backend avant d'importer keras (sinon par défaut c'est 'tensorflow'):
os.environ["KERAS_BACKEND"] = "torch"
# Memory optimization: Enable expandable segments to reduce fragmentation:
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
# Limiter la verbosité des logs de tensorflow:
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

from functions_model_building import (
    BREED_ID_SPLITTER_REGEX_PTRN,
    MyKerasSequence,
    plot_confusion_matrix,
)
from pathlib import Path
from sklearn.preprocessing import LabelEncoder

import datetime as dt
import itertools as it
import keras
import numpy as np
import pandas as pd
import pytz
import logging
logging.getLogger('tensorflow').setLevel(logging.ERROR)
import warnings
warnings.filterwarnings("ignore")

LOGGER_FORMAT = '%(asctime)s [%(levelname)s] %(message)s'
logging.Formatter.converter = lambda *_: dt.datetime.now(pytz.timezone('Europe/Paris')).timetuple()
logging.basicConfig(level=logging.INFO, format=LOGGER_FORMAT, force=True)
logr = logging.getLogger(__name__)
logr.setLevel(logging.DEBUG)

## Préparer les données en entrée

En entrée, un ou plusieurs chemins vers des répertoires contenant des images à classer

In [2]:
INPUT_PATHS_TO_DIRS = tuple([
    'images/n02090379-redbone',
    'images/n02085936-Maltese_dog',
    'images/n02088094-Afghan_hound',
])
input_img_paths = tuple(Path(dirname) for dirname in INPUT_PATHS_TO_DIRS)
input_img_paths = tuple(path.glob('*.jpg') for path in input_img_paths)
input_img_paths = tuple(it.chain.from_iterable(input_img_paths))
breed_labels = tuple(str(path).split('/')[1] for path in input_img_paths)
breed_labels_encoded = LabelEncoder().fit_transform(breed_labels)
breed_labels_dict = dict(sorted(zip(breed_labels_encoded, breed_labels)))

## Charger le modèle

In [3]:
PATH_TO_MODEL = (
    'models/from_pretrained/'
    # 'CNN__from=EfficientNetB0__n_cls=3__n_eps=20__drpt=0.20__LR=1e-05_model.keras'
    # 'CNN__from=EfficientNetV2B0__n_cls=3__n_eps=20__drpt=0.20__LR=1e-05_model.keras'
    'CNN__from=VGG16__n_cls=3__n_eps=20__drpt=0.20__LR=1e-05_model.keras'
)
if not os.path.exists(PATH_TO_MODEL):
    raise FileNotFoundError(PATH_TO_MODEL)
logr.info(f"Désérialiser l'objet sauvegardé {PATH_TO_MODEL!r}")
model = keras.saving.load_model(PATH_TO_MODEL)

2026-05-14 02:40:08,958 [INFO] Désérialiser l'objet sauvegardé 'models/from_pretrained/CNN__from=VGG16__n_cls=3__n_eps=20__drpt=0.20__LR=1e-05_model.keras'


## Prédire

Effectuer l'inférence

In [4]:
PREPROCESSING_FUNC = {
    'EfficientNetB0': keras.applications.efficientnet.preprocess_input,
    'EfficientNetV2B0': keras.applications.efficientnet_v2.preprocess_input,
    'VGG16': keras.applications.vgg16.preprocess_input,
}
input_img_paths = np.random.choice(input_img_paths, size=DEFAULT_BATCH_SIZE, replace=False)
test_seq = MyKerasSequence(
    paths=input_img_paths,
    labels=[''] * len(input_img_paths),
    batch_size=DEFAULT_BATCH_SIZE,
    target_size=DEFAULT_TRGT_IMG_SIZE,
    preprocessing_func=PREPROCESSING_FUNC['VGG16']
)
predictions = model.predict(test_seq)
predicted_class_idx = np.argmax(predictions, axis=1)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 438ms/step


## Afficher les résultats sous une forme conviviale

In [5]:
df = pd.DataFrame({
    'path': input_img_paths,
    'predicted_breed': [BREED_ID_SPLITTER_REGEX_PTRN.split(breed_labels_dict[class_idx])[1]
                        for class_idx in predicted_class_idx]
}, dtype=str)
df['is_correct'] = df.apply(
    lambda row: '✅' if row['predicted_breed'] in row['path'] else '',
    axis=1
)
df

,path,predicted_breed,is_correct
0,images/n02085936-Maltese_dog/n02085936_2020.jpg,Maltese_dog,✅
1,images/n02090379-redbone/n02090379_2463.jpg,redbone,✅
2,images/n02088094-Afghan_hound/n02088094_6035.jpg,Afghan_hound,✅
3,images/n02088094-Afghan_hound/n02088094_1370.jpg,Afghan_hound,✅
4,images/n02090379-redbone/n02090379_4632.jpg,redbone,✅
5,images/n02088094-Afghan_hound/n02088094_5150.jpg,Afghan_hound,✅
6,images/n02090379-redbone/n02090379_4150.jpg,redbone,✅
7,images/n02088094-Afghan_hound/n02088094_5517.jpg,Afghan_hound,✅
8,images/n02088094-Afghan_hound/n02088094_10832.jpg,Afghan_hound,✅
9,images/n02088094-Afghan_hound/n02088094_7131.jpg,Afghan_hound,✅
